# Queueing theory - M/M/n queueing table using Dask

This notebook produces the typical 'average waiting time in units of service time' table ($W_q / E[ST]$) for an M/M/n queue, for 1 to 10 berths (servers) $n$ and berth occupancies (server utilisations) $\rho$ of 10 % to 90 %, as presented in for example the PIANC (2014) guideline. Every combination is simulated with 1,000,000 customers to approach steady-state values.

Every combination is simulated as `nr_replications` independent simulations (Dask tasks) of `nr_arr` customers, so `nr_arr * nr_replications` customers in total per table cell:
* one simulation of 100,000 customers takes about 3 s and about 350 MB of memory, so many workers can run side by side (a single simulation of 1 million customers would need several GB per worker);
* 90 cells x 10 replications = 900 tasks that Dask spreads evenly over all workers;
* the spread between the replications gives the statistical uncertainty of the table values.

Each simulation starts with an empty system; with 100,000 customers per simulation the effect of this start-up period on the averages is negligible. At high occupancies successive waiting times are strongly correlated, so these cells converge slowest.

The total computation time is roughly (number of cells) x (customers per cell) x 27 µs / (number of workers): for 90 cells with 1 million customers each about 40 minutes of CPU time, i.e. a few minutes on 12 local workers.

The calculations are distributed with [Dask](https://docs.dask.org/en/stable/deploying.html):
* locally, a `LocalCluster` starts several worker *processes* on your computer (processes rather than threads, because the simulation is pure Python and threads would be blocked by Python's global interpreter lock);
* to upscale, the same code can connect to a larger cluster (e.g. a HPC or cloud cluster) by replacing the `LocalCluster` (see section 2).

NB: Dask is an optional dependency. Install it in your environment with `pip install "dask[distributed,diagnostics]"` (or `pip install -e ".[dask]"` from the OpenQTSim folder).

In [1]:
# OpenQTSim parallel engine
import os
import openqtsim
from openqtsim.mt_engine import create_tasks, run_tasks, waiting_time_table, mmc_waiting_time_factor
import numpy as np
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
from dask.distributed import Client, LocalCluster

print("This notebook is executed with OpenQTSim version {}".format(openqtsim.__version__))

This notebook is executed with OpenQTSim version v0.5.1


#### 0. Define input

In [2]:
A = "M"              # distribution of interarrival times (M: exponential, E2: erlang 2, ...)
S = "M"              # distribution of service times (M: exponential, E2: erlang 2, ...)
servers = range(1, 11)  # numbers of berths/servers n to consider
utilisations = np.round(np.arange(0.1, 1.0, 0.1), 2)  # berth occupancy rho = lambda / (n * mu_server)
mu = 10                 # total service capacity of the system (customers per hour); each server serves mu / n
customers_per_cell = 1_000_000  # customers per combination of n and rho
nr_arr = 100_000        # customers per simulation (task)
nr_replications = customers_per_cell // nr_arr  # independent simulations per table cell
seed = 2024             # makes the whole table reproducible
n_workers = os.cpu_count()  # number of local Dask worker processes

#### 1. Create task list
Each task is one simulation: a combination of berth occupancy, number of berths and replication, with its own independent arrival and service seeds.

In [3]:
tasks = create_tasks(A, S, servers, utilisations, nr_arr, mu=mu, nr_replications=nr_replications, seed=seed)

print(f"{len(tasks):,} simulations of {nr_arr:,} customers each ({nr_arr * nr_replications:,} customers per table cell)")
pd.DataFrame(tasks).head()

900 simulations of 100,000 customers each (1,000,000 customers per table cell)


,A,S,c,nr_arr,lam,mu,replication,arrival_seed,service_seed
0,M,M,1,100000,1.0,10,0,2855298535,925030728
1,M,M,1,100000,1.0,10,1,1146676384,1811149567
2,M,M,1,100000,1.0,10,2,524768821,2577401382
3,M,M,1,100000,1.0,10,3,4067285479,2609318523
4,M,M,1,100000,1.0,10,4,2651666590,174624609


#### 2. Start a Dask cluster
The client shows a link to the Dask dashboard (open it in your browser), where you can follow the workers and the progress of the tasks while they run.

To upscale beyond your own computer, replace the `LocalCluster` by a larger cluster; the rest of the notebook stays the same. For example on a SLURM cluster such as DelftBlue (with `pip install dask-jobqueue`):

```python
from dask_jobqueue import SLURMCluster
cluster = SLURMCluster(cores=8, processes=8, memory="16GB", walltime="01:00:00")
cluster.scale(jobs=10)  # 10 jobs x 8 worker processes = 80 workers
```

or connect to a running scheduler with `client = Client("tcp://scheduler-address:8786")`.

In [4]:
cluster = LocalCluster(n_workers=n_workers, threads_per_worker=1, processes=True)
client = Client(cluster)
client

Connection method: Cluster object,Cluster type: distributed.LocalCluster
Dashboard: http://127.0.0.1:8787/status,
Dashboard: http://127.0.0.1:8787/status,Workers: 12
Total threads: 12,Total memory: 16.00 GiB
Status: running,Using processes: True
Comm: tcp://127.0.0.1:58295,Workers: 0
Dashboard: http://127.0.0.1:8787/status,Total threads: 0
Started: Just now,Total memory: 0 B
Comm: tcp://127.0.0.1:58322,Total threads: 1
Dashboard: http://127.0.0.1:58323/status,Memory: 1.33 GiB
Nanny: tcp://127.0.0.1:58298,


2026-10-18 22:43:00,017 - tornado.application - ERROR - Uncaught exception GET /status/ws (127.0.0.1)
HTTPServerRequest(protocol='http', host='127.0.0.1:8787', method='GET', uri='/status/ws', version='HTTP/1.1', remote_ip='127.0.0.1')
Traceback (most recent call last):
  File "/opt/anaconda3/envs/OpenQTSim/lib/python3.13/site-packages/tornado/websocket.py", line 965, in _accept_connection
    open_result = handler.open(*handler.open_args, **handler.open_kwargs)
  File "/opt/anaconda3/envs/OpenQTSim/lib/python3.13/site-packages/tornado/web.py", line 3546, in wrapper
    return method(self, *args, **kwargs)
  File "/opt/anaconda3/envs/OpenQTSim/lib/python3.13/site-packages/bokeh/server/views/ws.py", line 167, in open
    raise ProtocolError("Token is expired. Configure the app with a larger value for --session-token-expiration if necessary")
bokeh.protocol.exceptions.ProtocolError: Token is expired. Configure the app with a larger value for --session-token-expiration if necessary


#### 3. Run the simulations in parallel

In [5]:
%%time
results = run_tasks(tasks, client)

CPU times: user 52.6 s, sys: 19.1 s, total: 1min 11s
Wall time: 6min 45s


#### 4. Waiting time table
Average waiting time in units of average service time ($W_q / E[ST]$) for the M/M/n queue, averaged over the replications:

In [6]:
def pianc_layout(df):
    """Rows: berth occupancy in %, columns: number of berths n."""
    df = df.copy()
    df.index = [f"{u:.0%}" for u in df.index]
    df.index.name = "Berth occup. (ρ/n)"
    df.columns.name = "Number of berths n"
    return df

table = waiting_time_table(tasks, results)
pianc_layout(table).style.format("{:.4f}")

Number of berths n,1,2,3,4,5,6,7,8,9,10
Berth occup. (ρ/n),,,,,,,,,,
10%,0.1109,0.0101,0.0013,0.0002,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000
20%,0.2483,0.0414,0.0102,0.0030,0.0010,0.0003,0.0001,0.0000,0.0000,0.0000
30%,0.4262,0.0990,0.0334,0.0134,0.0058,0.0027,0.0012,0.0007,0.0003,0.0002
40%,0.6700,0.1903,0.0777,0.0378,0.0197,0.0113,0.0065,0.0041,0.0025,0.0015
50%,1.0027,0.3303,0.1590,0.0862,0.0521,0.0330,0.0217,0.0148,0.0105,0.0071
60%,1.5055,0.5664,0.2989,0.1786,0.1182,0.0807,0.0594,0.0430,0.0324,0.0252
70%,2.3194,0.9589,0.5504,0.3544,0.2561,0.1893,0.1412,0.1095,0.0902,0.0711
80%,4.0136,1.8145,1.0866,0.7527,0.5412,0.4406,0.3514,0.2851,0.2374,0.2056
90%,9.0358,4.1721,2.6539,2.0354,1.4929,1.1576,1.0286,0.9056,0.7550,0.6982


Standard error of the table values (standard deviation between the replications divided by $\sqrt{\text{nr\_replications}}$):

In [7]:
std_error = waiting_time_table(tasks, results, aggfunc="std") / np.sqrt(nr_replications)
pianc_layout(std_error).style.format("{:.4f}")

Number of berths n,1,2,3,4,5,6,7,8,9,10
Berth occup. (ρ/n),,,,,,,,,,
10%,0.0007,0.0001,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000
20%,0.0012,0.0003,0.0002,0.0001,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000
30%,0.0010,0.0006,0.0003,0.0003,0.0001,0.0001,0.0000,0.0000,0.0000,0.0000
40%,0.0032,0.0014,0.0008,0.0004,0.0004,0.0002,0.0002,0.0002,0.0001,0.0001
50%,0.0063,0.0018,0.0016,0.0009,0.0010,0.0005,0.0004,0.0003,0.0003,0.0002
60%,0.0055,0.0048,0.0019,0.0012,0.0011,0.0008,0.0008,0.0007,0.0006,0.0004
70%,0.0189,0.0067,0.0060,0.0031,0.0036,0.0039,0.0011,0.0010,0.0014,0.0010
80%,0.0214,0.0120,0.0178,0.0105,0.0135,0.0072,0.0040,0.0040,0.0053,0.0030
90%,0.2167,0.0616,0.0568,0.0510,0.0416,0.0212,0.0277,0.0238,0.0140,0.0180


#### 5. Compare with the analytical M/M/n solution
For exponential interarrival and service times (M/M/n) the exact solution follows from the Erlang C formula: $W_q / E[ST] = C(n, n\rho) / (n(1-\rho))$, with $C$ the probability that an arriving customer has to wait and $\rho$ the berth occupancy. This reproduces the well-known PIANC (2014) table.

The differences are shown as absolute values (simulated - analytical), because relative differences are meaningless for the very small waiting times at low occupancies and many berths.

In [8]:
reference_name = "analytical (M/M/n)"
reference = pd.DataFrame(
    {n: [mmc_waiting_time_factor(n, u) for u in table.index] for n in table.columns}, index=table.index
)

difference = (table - reference).reindex(index=table.index, columns=table.columns)
display(pianc_layout(reference).style.format("{:.4f}", na_rep="").set_caption(reference_name))
display(pianc_layout(difference).style.format("{:+.4f}", na_rep="").set_caption(f"Simulated - {reference_name}"))

Number of berths n,1,2,3,4,5,6,7,8,9,10
Berth occup. (ρ/n),,,,,,,,,,
10%,0.1111,0.0101,0.0014,0.0002,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000
20%,0.2500,0.0417,0.0103,0.0030,0.0010,0.0003,0.0001,0.0000,0.0000,0.0000
30%,0.4286,0.0989,0.0333,0.0132,0.0058,0.0027,0.0013,0.0006,0.0003,0.0002
40%,0.6667,0.1905,0.0784,0.0378,0.0199,0.0111,0.0064,0.0039,0.0024,0.0015
50%,1.0000,0.3333,0.1579,0.0870,0.0521,0.0330,0.0218,0.0148,0.0102,0.0072
60%,1.5000,0.5625,0.2956,0.1794,0.1181,0.0819,0.0589,0.0436,0.0330,0.0253
70%,2.3333,0.9608,0.5470,0.3572,0.2519,0.1867,0.1432,0.1128,0.0906,0.0739
80%,4.0000,1.7778,1.0787,0.7455,0.5541,0.4315,0.3471,0.2860,0.2401,0.2046
90%,9.0000,4.2632,2.7235,1.9694,1.5250,1.2335,1.0285,0.8769,0.7606,0.6687


Number of berths n,1,2,3,4,5,6,7,8,9,10
Berth occup. (ρ/n),,,,,,,,,,
10%,-0.0003,-0.0000,-0.0000,-0.0000,+0.0000,+0.0000,-0.0000,-0.0000,-0.0000,+0.0000
20%,-0.0017,-0.0003,-0.0001,-0.0000,+0.0000,-0.0000,-0.0000,+0.0000,+0.0000,-0.0000
30%,-0.0023,+0.0001,+0.0001,+0.0002,+0.0000,+0.0001,-0.0001,+0.0000,-0.0000,+0.0000
40%,+0.0033,-0.0002,-0.0008,+0.0000,-0.0002,+0.0002,+0.0000,+0.0003,+0.0001,+0.0001
50%,+0.0027,-0.0030,+0.0011,-0.0008,-0.0001,-0.0000,-0.0001,+0.0000,+0.0003,-0.0001
60%,+0.0055,+0.0039,+0.0033,-0.0008,+0.0001,-0.0012,+0.0005,-0.0006,-0.0005,-0.0001
70%,-0.0139,-0.0019,+0.0033,-0.0028,+0.0042,+0.0027,-0.0020,-0.0032,-0.0004,-0.0028
80%,+0.0136,+0.0368,+0.0079,+0.0072,-0.0129,+0.0091,+0.0043,-0.0009,-0.0027,+0.0010
90%,+0.0358,-0.0910,-0.0696,+0.0660,-0.0320,-0.0759,+0.0001,+0.0287,-0.0056,+0.0295


In [9]:
fig = go.Figure()
colors = px.colors.qualitative.Plotly
for i, n in enumerate(table.columns):
    color = colors[i % len(colors)]
    fig.add_trace(go.Scatter(
        x=100 * table.index, y=table[n], error_y=dict(type="data", array=std_error[n]),
        mode="markers", marker=dict(color=color, size=8), name=f"n = {n} simulated", legendgroup=str(n),
    ))
    if n in reference.columns:
        fig.add_trace(go.Scatter(
            x=100 * reference.index, y=reference[n], mode="lines", line=dict(color=color),
            name=f"n = {n} {reference_name}", legendgroup=str(n),
        ))
fig.update_layout(
    title=f"{A}/{S}/n: average waiting time in units of service time ({nr_arr * nr_replications:,} customers per cell)",
    xaxis_title="berth occupancy ρ/n [%]", yaxis_title="W_q / E[ST] [-]", yaxis_type="log",
    height=600, template="plotly_white",
)
fig.show()

#### 6. Close the Dask cluster

In [10]:
client.close()
cluster.close()